# Whisper ASR Benchmark for Bororo Speech

This companion notebook transcribes the generated audio and computes Word Error Rate (WER) and Character Error Rate (CER) against the manifest. Set `OPENAI_API_KEY` in the environment; never store an API key in the notebook.

> The repository contains the benchmark workflow, but no aggregate WER/CER claim is reported until the full experiment is rerun on an authorized dataset.

In [ ]:
# Installa le librerie necessarie:
# - openai per chiamare le API Whisper
# - jiwer per calcolare WER e CER
# - soundfile per ottenere la durata degli audio
!pip -q install openai jiwer soundfile

In [ ]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter OPENAI_API_KEY (input is hidden): "
    )

In [ ]:
# Questa cella contiene TUTTO il codice del benchmark.
# In particolare:
# - normalizzazione e tokenizzazione del testo
# - caricamento del dizionario BOE
# - calcolo lexical coverage e OOV
# - (opzionale) calcolo WER e CER
# - chiamata alle API Whisper (whisper-1)
# - salvataggio dei risultati (CSV, JSONL, summary)

# È una versione "Colab-ready":
# - non usa argparse
# - espone una funzione run_benchmark(...) facile da chiamare

# ⚠️ NON esegue nulla: definisce solo funzioni e classi
# L'esecuzione vera avviene nella cella 5

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
AUDIO_DIR = PROJECT_ROOT / "artifacts/mms_bororo_audio"
MANIFEST_PATH = AUDIO_DIR / "manifest.csv"
DICTIONARY_PATH = PROJECT_ROOT / "data/boe-dict-20240805.json"
OUTPUT_DIR = PROJECT_ROOT / "artifacts/whisper_benchmark"

for required_path in (AUDIO_DIR, MANIFEST_PATH, DICTIONARY_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input not found: {required_path}")

In [ ]:
# No archive upload is needed: the companion generation notebook writes
# audio files and manifest.csv directly under artifacts/.
print(f"Audio directory: {AUDIO_DIR}")

In [ ]:
# The generation notebook already writes manifest.csv.
# Inspect a few manifest rows before starting the paid API benchmark.
import csv

with MANIFEST_PATH.open(encoding="utf-8") as manifest_file:
    manifest_preview = list(csv.DictReader(manifest_file))[:5]
manifest_preview

In [ ]:
# Questa cella contiene tutto lo script di benchmark,
# aggiornato per essere usato in Colab.
# Non esegue nulla finché non chiami run_benchmark().

from __future__ import annotations
import csv, json, os, re, sys, time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

try:
    import soundfile as sf
except:
    sf = None

try:
    from jiwer import wer, cer
except:
    wer = None
    cer = None

from openai import OpenAI

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".mpga", ".mpeg", ".mp4", ".ogg", ".webm", ".flac"}
_WORD_RE = re.compile(r"[^\w\s'’-]+", flags=re.UNICODE)

def normalize_text(s: str) -> str:
    s = (s or "").strip().lower()
    s = _WORD_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(s: str) -> List[str]:
    s = normalize_text(s)
    return s.split() if s else []

def load_dictionary(path: Path) -> Dict[str, str]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    norm = {}
    for k, v in data.items():
        nk = normalize_text(str(k))
        if nk:
            norm[nk] = str(v)
    return norm

def lexicon_coverage(tokens: List[str], lexicon: Dict[str, str], max_ngram: int = 4) -> Tuple[float, List[str]]:
    if not tokens:
        return 0.0, []
    covered = [False]*len(tokens)
    for i in range(len(tokens)):
        if covered[i]:
            continue
        for n in range(max_ngram, 1, -1):
            if i+n<=len(tokens):
                phrase=" ".join(tokens[i:i+n])
                if phrase in lexicon:
                    for j in range(i,i+n):
                        covered[j]=True
                    break
    for i,t in enumerate(tokens):
        if not covered[i] and t in lexicon:
            covered[i]=True
    cov=sum(covered)/len(tokens)
    oov=[t for i,t in enumerate(tokens) if not covered[i]]
    return cov,oov

def load_manifest(manifest_path: Path) -> Dict[str,str]:
    refs={}
    if manifest_path.suffix.lower()==".csv":
        with manifest_path.open("r",encoding="utf-8") as f:
            reader=csv.DictReader(f)
            for row in reader:
                a=(row.get("audio") or "").strip()
                t=(row.get("transcript") or "").strip()
                if a:
                    refs[Path(a).name]=t
    else:
        with manifest_path.open("r",encoding="utf-8") as f:
            for l in f:
                obj=json.loads(l.strip())
                a=str(obj.get("audio","")).strip()
                t=str(obj.get("transcript","")).strip()
                if a:
                    refs[Path(a).name]=t
    return refs

def get_audio_duration_seconds(path: Path) -> Optional[float]:
    if sf is None:
        return None
    try:
        info=sf.info(str(path))
        if info.frames and info.samplerate:
            return info.frames/info.samplerate
    except:
        return None
    return None

def transcribe_with_openai(client:OpenAI,audio_path:Path,model:str="whisper-1",language:Optional[str]=None,prompt:Optional[str]=None,temperature:Optional[float]=None) -> str:
    with audio_path.open("rb") as f:
        kwargs={"model":model,"file":f}
        if language: kwargs["language"]=language
        if prompt: kwargs["prompt"]=prompt
        if temperature is not None: kwargs["temperature"]=float(temperature)
        resp=client.audio.transcriptions.create(**kwargs)
        return str(getattr(resp,"text","") or "")

@dataclass
class ItemResult:
    audio_file: str
    duration_s: Optional[float]
    latency_s: float
    transcript: str
    transcript_norm: str
    n_words: int
    lexicon_coverage: float
    oov_count: int
    oov_sample: List[str]
    wer: Optional[float]
    cer: Optional[float]

def iter_audio_files(audio_dir: Path) -> Iterable[Path]:
    for p in sorted(audio_dir.rglob("*")):
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS:
            yield p

def run_benchmark(audio_dir:str, dict_json:str, out_dir:str="benchmark_out", manifest:Optional[str]=None, model:str="whisper-1", language:Optional[str]=None, prompt:Optional[str]=None, temperature:Optional[float]=None, max_ngram:int=4, limit:Optional[int]=None):
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY non impostata")

    audio_dir_p=Path(audio_dir)
    dict_json_p=Path(dict_json)
    out_dir_p=Path(out_dir)

    if not audio_dir_p.exists(): raise FileNotFoundError(f"{audio_dir} non trovato")
    if not dict_json_p.exists(): raise FileNotFoundError(f"{dict_json} non trovato")

    lexicon=load_dictionary(dict_json_p)
    refs=load_manifest(Path(manifest)) if manifest else {}
    client=OpenAI()

    out_dir_p.mkdir(exist_ok=True,parents=True)
    out_csv  = out_dir_p/"results.csv"
    out_jsonl= out_dir_p/"results.jsonl"
    out_summary=out_dir_p/"summary.json"

    audio_files=list(iter_audio_files(audio_dir_p))
    if limit is not None: audio_files=audio_files[:limit]
    if not audio_files: raise RuntimeError("Nessun audio trovato")

    results=[]
    for idx,audio_path in enumerate(audio_files, start=1):
        dur=get_audio_duration_seconds(audio_path)
        t0=time.perf_counter()
        transcript=transcribe_with_openai(client,audio_path,model,language,prompt,temperature)
        latency=time.perf_counter()-t0
        tnorm=normalize_text(transcript)
        toks=tokenize(transcript)
        cov,oov=lexicon_coverage(toks,lexicon,max_ngram)
        ref=refs.get(audio_path.name)
        item_wer=None
        item_cer=None
        if ref and wer and cer:
            item_wer=float(wer(normalize_text(ref),tnorm))
            item_cer=float(cer(normalize_text(ref),tnorm))
        results.append(ItemResult(str(audio_path),dur,latency,transcript,tnorm,len(toks),cov,len(oov),oov[:25],item_wer,item_cer))
        print(f"[{idx}/{len(audio_files)}] {audio_path.name} lat={latency:.2f}s words={len(toks)} cov={cov:.2%} oov={len(oov)}")

    with out_csv.open("w",encoding="utf-8",newline="") as f:
        writer=csv.DictWriter(f,fieldnames=list(asdict(results[0]).keys()))
        writer.writeheader()
        for r in results:
            row=asdict(r)
            row["oov_sample"]=" ".join(row["oov_sample"])
            writer.writerow(row)

    with out_jsonl.open("w",encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(asdict(r),ensure_ascii=False)+"\n")

    def avg(xs): return sum(xs)/len(xs) if xs else None
    latencies=[r.latency_s for r in results]
    coverages=[r.lexicon_coverage for r in results if r.n_words>0]
    oov_rates=[(r.oov_count/r.n_words) for r in results if r.n_words>0]
    wers=[r.wer for r in results if r.wer is not None]
    cers=[r.cer for r in results if r.cer is not None]

    summary={
        "n_files":len(results),"model":model,"audio_dir":audio_dir,"dict_json":dict_json,
        "manifest":manifest,"avg_latency_s":avg(latencies),"avg_lexicon_coverage":avg(coverages),
        "avg_oov_rate":avg(oov_rates),"avg_wer":avg(wers) if wers else None,"avg_cer":avg(cers) if cers else None,
        "notes":{"duration_available":sf is not None,"wer_cer_available":bool(refs and wer and cer)}
    }
    with out_summary.open("w",encoding="utf-8") as f:
        json.dump(summary,f,ensure_ascii=False,indent=2)

    print("\n--- OUTPUT ---\nCSV:",out_csv,"\nJSONL:",out_jsonl,"\nSUMMARY:",out_summary)
    return out_dir_p

In [ ]:
# Start with a small limit because API transcription has a monetary cost.
out_dir = run_benchmark(
    audio_dir=str(AUDIO_DIR),
    dict_json=str(DICTIONARY_PATH),
    out_dir=str(OUTPUT_DIR),
    manifest=str(MANIFEST_PATH),
    language=None,  # Bororo is not a supported Whisper language hint
    prompt=None,
    temperature=0,
    max_ngram=4,
    limit=50,
)

In [ ]:
from pathlib import Path

for result_name in ("results.csv", "results.jsonl", "summary.json"):
    print(Path(out_dir) / result_name)